# 05 — Churn Definition and Label Design

A marketplace does not have an explicit cancellation event. We infer churn from purchase inactivity. The threshold must reflect the long and irregular purchase intervals in the data.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.data_loader import load_olist_data
from src.analytics import prepare_order_level
from src.churn import build_churn_snapshots

data = load_olist_data(ROOT / "data_raw")
customers = data["customers"]
orders = data["orders"]

customer_orders = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)



## Step 1 — Calculate inter-purchase gaps


In [ ]:
customer_orders = customer_orders.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"]
)

customer_orders["previous_purchase"] = (
    customer_orders.groupby("customer_unique_id")["order_purchase_timestamp"].shift(1)
)

customer_orders["purchase_gap_days"] = (
    customer_orders["order_purchase_timestamp"]
    - customer_orders["previous_purchase"]
).dt.total_seconds() / 86400

purchase_gaps = customer_orders["purchase_gap_days"].dropna()
purchase_gaps.describe(percentiles=[0.25, 0.5, 0.75, 0.8, 0.9, 0.95])



The full repeat-gap distribution included same-day / very short gaps. To focus on more meaningful behavioral returns, repeat gaps of seven days or less were removed for threshold selection.


In [ ]:
behavioral_gaps = purchase_gaps[purchase_gaps > 7]

behavioral_gaps.describe(
    percentiles=[0.5, 0.75, 0.8, 0.9, 0.95]
)



In [ ]:
for threshold in [180, 210, 240, 270]:
    coverage = (behavioral_gaps <= threshold).mean() * 100
    print(f"{threshold} days: {coverage:.2f}% of behavioral repeat gaps")



## Step 2 — Select the working inactivity threshold

Observed coverage from the practice run:

| Threshold | Behavioral repeat gaps within threshold |
|---:|---:|
| 180 days | 74.42% |
| 210 days | 79.96% |
| 240 days | 84.04% |
| 270 days | 87.68% |

**Selected working threshold: 240 days.** It captures about 84% of observed behavioral repeat gaps while avoiding the additional month of waiting associated with 270 days. This is a project-specific rule, not a universal marketplace standard.


## Step 3 — Construct leakage-aware historical snapshots

The target is defined for customers active in the previous 180 days:

- churned_240d = 1 when there is no delivered purchase in the next 240 days;
- churned_240d = 0 when at least one delivered purchase occurs in that period.

Features must use only purchases on or before each snapshot date. Only snapshots with a complete future 240-day observation period are retained.


In [ ]:
order_level = prepare_order_level(data)

analysis_end = pd.Timestamp("2018-08-31 23:59:59")

churn_snapshots = build_churn_snapshots(
    order_level,
    analysis_end=analysis_end,
    horizon_days=240,
    active_lookback_days=180,
    feature_lookback_days=180
)

churn_snapshots["churned_240d"].value_counts()



In [ ]:
churn_snapshots["churned_240d"].value_counts(normalize=True).mul(100).round(2)



The class distribution above must be generated by running the notebook on the local Olist files. It is not pre-filled because the canonical pipeline uses delivered orders, which may differ from the earlier all-status exploratory population.

**Leakage rule:** customer IDs, future purchases, future paid value and target columns are excluded from model features.


## Label validation against the complete archive

The 240-day no-return rule was implemented using only future delivered orders after each snapshot. In the chronological test snapshots, 98.27% of eligible customers were labeled churned. This extreme prevalence means the label is not sufficiently selective for retention targeting. The 240-day rule remains a working definition for this dataset, not a universal churn truth. A next iteration should separate first-time buyers from proven repeat buyers and test category-specific or survival-based definitions.

In [ ]:
import json
with open(ROOT / "outputs" / "churn_model_metrics.json", encoding="utf-8") as f:
    churn_metrics = json.load(f)
print({k: churn_metrics[k] for k in ["test_churn_prevalence", "roc_auc", "lift_at_10pct", "quality_note"]})